In [0]:
from datetime import datetime
container_name = dbutils.widgets.get("container_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
folder_name = dbutils.widgets.get("folder_name")
storage_account_key = dbutils.secrets.get(scope="amazon", key="storage_account_key")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
folder_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{folder_name}"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
#dbutils.fs.put(f"{folder_path}/{datetime.now()}.json", json.dumps(data), overwrite=True)

In [0]:
# items = dbutils.fs.ls(f'{folder_path}')
# loaded_files = [item.name for item in items]
if len(dbutils.fs.ls(folder_path)) == 0:
    dbutils.jobs.taskValues.set("run_now","false")
    dbutils.notebook.exit("No new files to process")
else:
    print("files are available")
    dbutils.jobs.taskValues.set("run_now","true")

In [0]:
spark.conf.set(
    "fs.azure.account.key.amazonapi.dfs.core.windows.net",
    storage_account_key
)

from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.functions import udf, col, regexp_extract, input_file_name
import urllib.parse

StructType_schema = StructType([
    StructField("AmazonOrderId", StringType()),
    StructField("PurchaseDate", StringType()),
    StructField("LastUpdateDate", StringType()),
    StructField("OrderStatus", StringType()),
    StructField("FulfillmentChannel", StringType()),
    StructField("SalesChannel", StringType()),
    StructField("ShipServiceLevel", StringType()),
    StructField("OrderTotal", StructType([
        StructField("CurrencyCode", StringType()),
        StructField("Amount", StringType())
    ])),
    StructField("NumberOfItemsShipped", StringType()),
    StructField("NumberOfItemsUnshipped", StringType()),
    StructField("PaymentMethod", StringType()),
    StructField("PaymentMethodDetails", ArrayType(StringType())),
    StructField("IsReplacementOrder", StringType()),
    StructField("MarketplaceId", StringType()),
    StructField("ShipmentServiceLevelCategory", StringType()),
    StructField("OrderType", StringType()),
    StructField("EarliestShipDate", StringType()),
    StructField("LatestShipDate", StringType()),
    StructField("EarliestDeliveryDate", StringType()),
    StructField("LatestDeliveryDate", StringType()),
    StructField("IsBusinessOrder", StringType()),
    StructField("IsPrime", StringType()),
    StructField("IsGlobalExpressEnabled", StringType()),
    StructField("IsPremiumOrder", StringType()),
    StructField("IsSoldByAB", StringType()),
    StructField("IsIBA", StringType()),
    StructField("IsISPU", StringType()),
    StructField("IsAccessPointOrder", StringType()),
    StructField("AutomatedShippingSettings", StructType([
        StructField("HasAutomatedShippingSettings", StringType())
    ])),
    StructField("EasyShipShipmentStatus", StringType()),
    StructField("ElectronicInvoiceStatus", StringType()),
    StructField("DefaultShipFromLocationAddress", StructType([
        StructField("Name", StringType()),
        StructField("AddressLine1", StringType()),
        StructField("City", StringType()),
        StructField("StateOrRegion", StringType()),
        StructField("PostalCode", StringType()),
        StructField("CountryCode", StringType()),
        StructField("Phone", StringType()),
        StructField("AddressType", StringType())
    ])),
    StructField("FulfillmentInstruction", StructType([
        StructField("FulfillmentSupplySourceId", StringType())
    ]))
])

df = spark.read.schema(StructType_schema).option("multiline", "true").option('header','true').json(folder_path)
display(df)

In [0]:
decode = udf(lambda x: urllib.parse.unquote(x), StringType())
df_flattened =  df.withColumn("OrderTotal_CurrencyCode", col("OrderTotal.CurrencyCode")) \
                 .withColumn("OrderTotal_Amount", col("OrderTotal.Amount")) \
                 .withColumn("DefaultShipFromLocationAddress_Name", col("DefaultShipFromLocationAddress.Name")) \
                 .withColumn("DefaultShipFromLocationAddress_AddressLine1", col("DefaultShipFromLocationAddress.AddressLine1")) \
                 .withColumn("DefaultShipFromLocationAddress_City", col("DefaultShipFromLocationAddress.City")) \
                 .withColumn("DefaultShipFromLocationAddress_StateOrRegion", col("DefaultShipFromLocationAddress.StateOrRegion")) \
                 .withColumn("DefaultShipFromLocationAddress_PostalCode", col("DefaultShipFromLocationAddress.PostalCode")) \
                 .withColumn("DefaultShipFromLocationAddress_CountryCode", col("DefaultShipFromLocationAddress.CountryCode")) \
                 .withColumn("DefaultShipFromLocationAddress_Phone", col("DefaultShipFromLocationAddress.Phone")) \
                 .withColumn("DefaultShipFromLocationAddress_AddressType", col("DefaultShipFromLocationAddress.AddressType")) \
                 .withColumn("FulfillmentIntruction_FulfillmentSupplySourceId", col("FulfillmentInstruction.FulfillmentSupplySourceId")) \
                 .withColumn("AutomatedShippingSettingsStatus", col("AutomatedShippingSettings.HasAutomatedShippingSettings")) \
                 .withColumn("PaymentMethodDetail", col("PaymentMethodDetails")[0]) \
                 .withColumn("file_name", decode(regexp_extract(input_file_name(), r"([^/]+$)", 1))) \
                 .drop("OrderTotal", "DefaultShipFromLocationAddress","FulfillmentInstruction", "AutomatedShippingSettings","PaymentMethodDetails")
                 
display(df_flattened)

In [0]:
df_flattened.write.format('delta').option("mergeSchema", "true").mode('append').saveAsTable(f'{catalog}.{schema}.amazon_orders_bronze')

In [0]:
%sql
select * from ${catalog}.${schema}.amazon_orders_bronze